In [1]:
# special for ribosome only, make sure to include code in building me model to not make ribosomes the same way 
# as other proteins

In [5]:
import cobra

import pandas as pd
import numpy as np

from Bio.Seq import Seq
from Bio import SeqIO

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
import build_mrna_expression_reactions
from build_mrna_expression_reactions import Transcript, human_model, rnap, seq_element_map, seq_metabolite_map, ppi_n, nmp_map_c, h2o_c, h_c, exosome, h2o_n, h_n, nmp_map_n, lariat_machinery, atp_n 


In [17]:
# variables needed
ntp_map_c = {'C': human_model.metabolites.get_by_id('ctp[c]'), 
             'U': human_model.metabolites.get_by_id('utp[c]'), 
             'G': human_model.metabolites.get_by_id('gtp[c]'), 
             'A': human_model.metabolites.get_by_id('atp[c]')}
ntp_map_n = {v:k for k,v in seq_metabolite_map.items()}

# 5s rrna
rnap3 = rnap[rnap['Approved name'].isin([i for i in rnap['Approved name'] if ' III ' in i])]
tfiiia, tfiiib = ['HGNC:4662'], ['HGNC:13652', 'HGNC:11551', 'HGNC:11588']
tfiiic = ['HGNC:4664', 'HGNC:4665', 'HGNC:4666', 'HGNC:4667', 'HGNC:4668', 'HGNC:20872']
five_s_transcription_machinery = rnap3['HGNC ID (gene)'].tolist() + tfiiia + tfiiib + tfiiic
REXO5 = 'HGNC:24661'
xpo1 = ['HGNC:12825']

# other rrnas
rnap1 = rnap[rnap['Approved name'].isin([i for i in rnap['Approved name'] if ' I ' in i])]
taf = ['HGNC:11532', 'HGNC:11533', 'HGNC:11534']
ubf = ['HGNC:12511']
rnap1_tfs = taf + ubf
UTP10 = ['HGNC:25517'] # a' cleavage
RNASEN = ['HGNC:17904'] # assumed site 02 endonucleolytic cleavage
RMRP = ['HGNC:10031']
UTP23 = ['HGNC:28224']
UTP24 = ['HGNC:20220']
PARN = ['HGNC:8609']
PAPD5 = ['HGNC:30758'] #TENT4B
NOB1 = ['HGNC:29540']
LAS1 = ['HGNC:25726']
DIS3 = ['HGNC:20604']
ISG20L2 = ['HGNC:25745']
ERI1 = ['HGNC:23994']

In [7]:
# rrna sequences
# assume the ncbi 45s is actually 47s...see notes for details
rrna_47s_seq = SeqIO.read(local_data_path + 'raw/45s_rrna_seq.txt', "fasta").seq.transcribe()
rrna_18s_seq = SeqIO.read(local_data_path + 'raw/18s_rrna_seq.txt', "fasta").seq.transcribe()
rrna_28s_seq = SeqIO.read(local_data_path + 'raw/28s_rrna_seq.txt', "fasta").seq.transcribe()
rrna_5_8s_seq = SeqIO.read(local_data_path + 'raw/5_8s_rrna_seq.txt', "fasta").seq.transcribe()
ets_5_seq = rrna_47s_seq[:rrna_47s_seq.index(rrna_18s_seq)]
its_1_seq = rrna_47s_seq[rrna_47s_seq.index(rrna_18s_seq) + len(rrna_18s_seq):rrna_47s_seq.index(rrna_5_8s_seq)]
its_2_seq = rrna_47s_seq[rrna_47s_seq.index(rrna_5_8s_seq) + len(rrna_5_8s_seq):rrna_47s_seq.index(rrna_28s_seq)]
ets_3_seq = rrna_47s_seq[rrna_47s_seq.index(rrna_28s_seq) + len(rrna_28s_seq):]

pre_rrna_5s_seq = SeqIO.read(local_data_path + 'raw/5s_rrna_seq.txt', "fasta").seq.transcribe()
rrna_5s_seq = pre_rrna_5s_seq[:120] # 120 is length of mature 5s_rrna



# rrna cut sites
# cut site indexes, relative to how far right (3' end is right) they are of a certain feature
# Fig. 3b https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4361047/
# scaled from length in figure to length of my sequence
A_prime_index = int(round(np.median([414,420])*len(ets_5_seq)/3657)) # location from 5' end of 47s
A_0_index = int(round(1642*len(ets_5_seq)/3657)) - A_prime_index # how far to the right of 45s is the A_0 site
site_2_index = int(round((6470-5527)*len(its_1_seq)/(6623-5527))) # how far right of end of 18s
site_4_index = int(round((7570-6779)*(len(its_2_seq)/(7935-6779)))) # how far to the right of the end of 5.8s/how far into ITS2 is the site 4 cut location
e_index = int(round((np.median([5606,5609])-5527)*len(its_1_seq)/(6623-5527)))# bp to right of end of 18s/start of its_1
conserved_stall_idx = int(round((np.median([6117,6192])-5527)*len(its_1_seq)/(6623-5527))) # how far right of end of 18s does RRP6 stall to form 21S-C
# Fig. 6b https://www.sciencedirect.com/science/article/pii/S1097276513005844?via%3Dihub
seven_s_idx = 190 
five_eight_plus_forty_idx = 40
six_s_index = 1 #https://www.nature.com/articles/s41594-019-0234-x?draft=collection

# # original: from 2+ difference soures
# # 420 and numerator from figure 3A https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3964915/
# A_prime_index = round(420*(len(ets_5_seq)/(1800 + 2000 + 420)))
# # Fig1D: https://www.researchgate.net/figure/Mapping-the-cleavages-in-human-ITS1-A-Alternative-processing-pathways-of-human-rRNA_fig1_235729322
# site_2_index = int(round(np.median([6396 -5520,6508-5520]) *(len(its_1_seq)/(6603-5520))))
# #https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3964915/
# A_0_index = round(1800*(len(ets_5_frag2_seq)/(1800 + 2000)))  
# # supplementary Fig. 2 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3632142/
# conserved_stall_idx = int(round(np.median([590, 635])))
# # Fig1D: https://www.researchgate.net/figure/Mapping-the-cleavages-in-human-ITS1-A-Alternative-processing-pathways-of-human-rRNA_fig1_235729322
# site_4_index = int(round((7564-6773)*(len(its_2_seq)/(7891 - 6773))))
# e_index = 80 # fig 3c:https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3632142/
# yeast_its2_length = 420 # https://www.microbiologyresearch.org/docserver/fulltext/jmm/66/2/126_jmm000426.pdf?expires=1594927728&id=id&accname=guest&checksum=7C6B2DF6CE3C3080E28605D15B99DF1E
# yeast_c2 = 140 # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4361047/
# seven_s_idx = int(round((yeast_c2/yeast_its2_length)*len(its_2_seq))) # how far to the right of 5.8s does sequence extend to form the 7s rrrna




In [8]:
rs = pd.read_csv(local_data_path + 'raw/small_ribosomal_protein.csv', index_col = None, skiprows = [0])
rl = pd.read_csv(local_data_path + 'raw/large_ribosomal_protein.csv', index_col = None, skiprows = [0])

In [9]:
def get_base_counts_and_elements(seq, triphosphate = True):
    '''Seq is a Bio.Seq object. Triphosphate is a boolean indicated whether the 5 end has a triphosphate. 
    Otherwise assume it is a monophosphate'''
    base_counts = dict()
    for base_letter in seq_element_map.keys():
        base_counts[base_letter] = seq.count(base_letter)
        
    elements = {'C': 0, 'H': 0, 'N': 0, 'O': 0, 'P': 0}
    for base_letter in seq_element_map.keys():
        for element in elements.keys():
            elements[element] += base_counts[base_letter]* seq_element_map[base_letter][element]   
    
    #3' end
    elements['H'] += 1 
    elements['O'] += 1
    
    # 5' end
    if triphosphate:
        elements['P'] += 2
        elements['O'] += 6
    else:
        elements['H'] += 1
      
        
    return base_counts, elements

def make_rrna_metabolite(name, seq, compartment = 'n', triphosphate = True):

    rrna_n = cobra.Metabolite(name + '_rrna[' + compartment + ']')
    rrna_n.compartment = compartment
    base_counts, elements = get_base_counts_and_elements(seq, triphosphate = triphosphate)

    rrna_n.elements = elements
    rrna_n.charge = -len(seq)
    
    if triphosphate:
        rrna_n.charge -= 3
    
    return rrna_n, base_counts


def fragment_degradation(fragment, fragment_base_counts, fragment_seq, name, triphosphate = True, nucleus = True):
    if nucleus: 
        fragment_degradation = cobra.Reaction(name + '_DEGRADATION')
        fragment_degradation.subsytem = 'Ribosome_Biogenesis'
        rxn = dict()
        rxn[h2o_n] = -sum(fragment_base_counts.values())+1
        rxn[fragment] = -1
        for k,v in nmp_map_n.items():
            rxn[v] = fragment_base_counts[k]

        # triphosphate on 5' end
        if triphosphate:
            rxn[nmp_map_n[fragment_seq[0]]] -= 1
            rxn[ntp_map_n[fragment_seq[0]]]  = 1  
            rxn[h_n] = sum(fragment_base_counts.values())-1
        else:
            rxn[h_n] = sum(fragment_base_counts.values()) # extra H on 5' end <--unsure about this

        fragment_degradation.add_metabolites(rxn)

        rule_part2 = ' and '.join(lariat_machinery['Exosome'] + lariat_machinery['NEXT Complex']) + ')'
        fragment_degradation.gene_reaction_rule = lariat_machinery["5' Degradation"][0] + ' or (' + rule_part2

        
    else:
        fragment_degradation = cobra.Reaction(name + '_DEGRADATION')
        fragment_degradation.subsytem = 'Ribosome_Biogenesis'
        rxn = dict()
        rxn[h2o_c] = -sum(fragment_base_counts.values())+1
        rxn[fragment] = -1
        for k,v in nmp_map_c.items():
            rxn[v] = fragment_base_counts[k]

        # triphosphate on 5' end
        if triphosphate:
            rxn[nmp_map_c[fragment_seq[0]]] -= 1
            rxn[ntp_map_c[fragment_seq[0]]]  = 1  
            rxn[h_c] = sum(fragment_base_counts.values())-1
        else:
            rxn[h_c] = sum(fragment_base_counts.values()) # extra H on 5' end <--unsure about this

        fragment_degradation.add_metabolites(rxn)

        fragment_degradation.gene_reaction_rule = ' and '.join(exosome['HGNC ID (gene)'].tolist())

        
    return fragment_degradation   


def rrna_degradation(rrna_c, rrna_base_counts, rrna_seq, name, triphosphate = True):
    '''RRNA_C is a cobra.Metabolite object representing mature, cytoplasmic rrna. rrna_base_counts is a dictionary
    mapping the number of NTPs per base.'''
    # https://www.cell.com/fulltext/S0092-8674(09)00067-1
    # https://pubmed.ncbi.nlm.nih.gov/18385160/
    # couple to ROS?
    
    rRNA_degradation = cobra.Reaction(name + '_rRNA_DEGRADATION')
    rRNA_degradation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_c] = -sum(rrna_base_counts.values())+1#+2 # 2 phosphodiester bonds
    rxn[h_c] = sum(rrna_base_counts.values())-1
    rxn[rrna_c] = -1
    for k,v in nmp_map_c.items():
        rxn[v] = rrna_base_counts[k]

    # triphosphate on 5' end
    if triphosphate:
        rxn[nmp_map_c[rrna_seq[0]]] -= 1
        rxn[ntp_map_c[rrna_seq[0]]]  = 1  

    rRNA_degradation.add_metabolites(rxn)
    rRNA_degradation.gene_reaction_rule = ' and '.join(exosome['HGNC ID (gene)'].tolist())
    
    return rRNA_degradation

In [10]:
def build_rrna5s_reactions():
    
    # TRANSCRIPTION - basically emulates Transcript.transcript_elongation reaction
    pre_rrna5s_n, pre_base_counts5s = make_rrna_metabolite('pre_5s', pre_rrna_5s_seq)

    rrna5s_transcription = cobra.Reaction('TRANSCRIPTION_PRE_RRNA5s')
    rrna5s_transcription.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    for ntp, base_letter in seq_metabolite_map.items():
        rxn[ntp] = -1*pre_base_counts5s[base_letter]
    rxn[ppi_n] = len(pre_rrna_5s_seq) - 1
    rxn[pre_rrna5s_n] = 1
    rrna5s_transcription.add_metabolites(rxn)
    rrna5s_transcription.gene_reaction_rule = ' and '.join(five_s_transcription_machinery) 
    
    # PROCESSING - mature rrna (3->5' exonucleolytic cleave of last 24 bases)
    rrna5s_processing = cobra.Reaction('PROCESSING_RRNA5s')
    rrna5s_processing.subsytem = 'Ribosome_Biogenesis'
    rrna5s_n, base_counts5s = make_rrna_metabolite('5s', rrna_5s_seq, compartment = 'n') 
    deg_base_counts = dict()
    for k,v in pre_base_counts5s.items():
        deg_base_counts[k] = v - base_counts5s[k]

    rxn = dict()
    rxn[h2o_n] = -sum(deg_base_counts.values()) # no -1 because all bonds 5'-most bond cleave
    rxn[h_n] = sum(deg_base_counts.values())
    for k,v in nmp_map_n.items():
        rxn[v] = deg_base_counts[k]


    rxn[pre_rrna5s_n] = -1
    rxn[rrna5s_n] = 1


    rrna5s_processing.add_metabolites(rxn)
    rrna5s_processing.gene_reaction_rule = REXO5

    
    # TRANSPORT 
    # must add nucleocytoplasmic export via ran gtp: https://www.sciencedirect.com/science/article/pii/S0171933504702575?via%3Dihub
    rrna5s_c = rrna5s_n.copy()
    rrna5s_c.id = rrna5s_c.id.replace('[n]', '[c]')
    rrna5s_c.compartment = 'c'
    
    rrna5s_transport = cobra.Reaction('RRNA5stn')
    rrna5s_transport.name = 'rRNA5s nuclear export'
    rrna5s_transport.subsytem = 'Ribosome_Biogenesis'
    rxn = {rrna5s_n: -1, rrna5s_c: 1}
    rrna5s_transport.add_metabolites(rxn)
    rrna5s_transport.gene_reaction_rule = ' and '.join(tfiiia + ['nucleocytoplasmic_export'])
    
    # Degradation
    rrna5s_degradation = rrna_degradation(rrna5s_c, base_counts5s, rrna_5s_seq, name = '5s')
   
    
    return rrna5s_transcription, rrna5s_processing, rrna5s_transport, rrna5s_degradation

In [14]:
# ets_5_frag1 is from 5' end of 47s to A' site
# ets_5_frag2 is from A' to 18s
# ets_5_frag3 is from A' to A0
# ets_5_frag4 is from A0 site to site 1 (start of 18s)
# its_1_frag1_seq is between site E and teh conserved stall location of RRP6
# its_1_frag2_seq is less than E site (some degradation) + a polyA/U tail

def build_other_rrna_reactions():
    # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6315592/ figure 2
    
    # 47s transcription------------------------------------------------------------------------------------
    rrna_47s_n, base_counts_47s = make_rrna_metabolite('47s', rrna_47s_seq)
    rrna_47s_transcription = cobra.Reaction('TRANSCRIPTION_RRNA_47s')
    rrna_47s_transcription.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    for ntp, base_letter in seq_metabolite_map.items():
        rxn[ntp] = -1*base_counts_47s[base_letter]
    rxn[ppi_n] = len(rrna_47s_seq) - 1
    rxn[rrna_47s_n] = 1
    rrna_47s_transcription.add_metabolites(rxn)
    rrna_47s_transcription.gene_reaction_rule = ' and '.join(rnap1['HGNC ID (gene)'].tolist() + rnap1_tfs)  
    
    # 45s formation------------------------------------------------------------------------------------
     
    ets_5_frag1_seq = ets_5_seq[:A_prime_index] 
    ets_5_frag2_seq = ets_5_seq[A_prime_index:]
    rrna_45s_seq = rrna_47s_seq[A_prime_index:rrna_47s_seq.index(rrna_28s_seq) + len(rrna_28s_seq)]

    ets_5_frag1_n, base_counts_ets_5_frag1 = make_rrna_metabolite('ets_5_frag1', ets_5_frag1_seq)
    rrna_45s_n, base_counts_rrna_45s = make_rrna_metabolite('45s', rrna_45s_seq, triphosphate=False)
    ets_3_n, base_counts_ets_3 = make_rrna_metabolite('ets_3', ets_3_seq, triphosphate = False)

    rrna_45s_formation = cobra.Reaction('FORMATION_RRNA_45s')
    rrna_45s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -2 # 2 endonuclolytic cleavage events to go from 47s to 45s
    rxn[rrna_47s_n], rxn[rrna_45s_n], rxn[ets_3_n], rxn[ets_5_frag1_n] = -1, 1, 1, 1
    rrna_45s_formation.add_metabolites(rxn)
    rrna_45s_formation.gene_reaction_rule = ' and '.join(UTP10 + RNASEN)
    
    ets_3_degradation = fragment_degradation(ets_3_n, base_counts_ets_3, ets_3_seq, name = 'ets_3', triphosphate = False)
    ets_5_frag1_degradation = fragment_degradation(ets_5_frag1_n, base_counts_ets_5_frag1, ets_5_frag1_seq, name = 'ets_5_frag1', triphosphate = True)

    #45S-->30S + 32.5S------------------------------------------------------------------------------------

    idx_30s = rrna_45s_seq.index(rrna_18s_seq) + len(rrna_18s_seq)
    rrna_30s_seq = ets_5_frag2_seq + rrna_18s_seq + rrna_45s_seq[idx_30s:idx_30s + site_2_index]
    rrna_32_5s_seq = rrna_45s_seq[idx_30s + site_2_index:]

    rrna_30s_n, base_counts_rrna_30s = make_rrna_metabolite('30s', rrna_30s_seq, triphosphate=False)
    rrna_32_5s_n, base_counts_rrna_32_5s = make_rrna_metabolite('32_5s', rrna_32_5s_seq, triphosphate=False)

    rrna_30s_formation = cobra.Reaction('FORMATION_RRNA_30s_32_5s')
    rrna_30s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site 2
    rxn[rrna_45s_n], rxn[rrna_30s_n], rxn[rrna_32_5s_n] = -1, 1, 1
    rrna_30s_formation.add_metabolites(rxn)
    rrna_30s_formation.gene_reaction_rule = RMRP[0]
    
    #26s formation------------------------------------------------------------------------------------
    
    rrna_26s_seq = ets_5_frag2_seq[A_0_index:] + rrna_18s_seq + its_1_seq[:site_2_index]
    ets_5_frag3_seq = ets_5_frag2_seq[:A_0_index]
    rrna_26s_n, base_counts_rrna_26s = make_rrna_metabolite('26s', rrna_26s_seq, triphosphate=False)
    ets_5_frag3_n, base_counts_ets_5_frag3 = make_rrna_metabolite('ets_5_frag3', ets_5_frag3_seq, triphosphate=False)

    rrna_26s_formation = cobra.Reaction('FORMATION_RRNA_26s')
    rrna_26s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site 2
    rxn[rrna_30s_n], rxn[rrna_26s_n], rxn[ets_5_frag3_n] = -1, 1, 1
    rrna_26s_formation.add_metabolites(rxn)
    rrna_26s_formation.gene_reaction_rule = UTP23[0]

    ets_5_frag3_degradation = fragment_degradation(ets_5_frag3_n, base_counts_ets_5_frag3, ets_5_frag3_seq, name = 'ets_5_frag3', triphosphate = False)

    # 21S formation------------------------------------------------------------------------------------
    rrna_21s_seq = rrna_18s_seq + its_1_seq[:site_2_index]
    ets_5_frag4_seq = ets_5_frag2_seq[A_0_index:]
    rrna_21s_n, base_counts_rrna_21s = make_rrna_metabolite('21s', rrna_21s_seq, triphosphate=False)
    ets_5_frag4_n, base_counts_ets_5_frag4 = make_rrna_metabolite('ets_5_frag4', ets_5_frag4_seq, triphosphate=False)

    rrna_21s_formation = cobra.Reaction('FORMATION_RRNA_21s')
    rrna_21s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site 1
    rxn[rrna_26s_n], rxn[rrna_21s_n], rxn[ets_5_frag4_n] = -1, 1, 1
    rrna_21s_formation.add_metabolites(rxn)
    rrna_21s_formation.gene_reaction_rule = UTP24[0]
    ets_5_frag4_degradation = fragment_degradation(ets_5_frag4_n, base_counts_ets_5_frag4, ets_5_frag4_seq, name = 'ets_5_frag4', triphosphate = False)

    # 21SC formation------------------------------------------------------------------------------------

    rrna_21sc_seq = rrna_18s_seq + its_1_seq[:conserved_stall_idx]
    rrna_21sc_n, base_counts_rrna_21sc = make_rrna_metabolite('21sc', rrna_21sc_seq, triphosphate=False)

    deg_seq = its_1_seq[conserved_stall_idx: site_2_index]
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_21sc_formation = cobra.Reaction('FORMATION_RRNA_21sc')
    rrna_21sc_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[rrna_21s_n], rxn[rrna_21sc_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)
    rrna_21sc_formation.add_metabolites(rxn)
    rrna_21sc_formation.gene_reaction_rule = exosome[exosome['Approved symbol'] == 'EXOSC10']['HGNC ID (gene)'].tolist()[0]

    # 18se formation------------------------------------------------------------------------------------
    rrna_18se_seq = rrna_18s_seq + its_1_seq[:e_index]
    its_1_frag1_seq = its_1_seq[e_index:conserved_stall_idx]
    rrna_18se_n, base_counts_rrna_18se = make_rrna_metabolite('18se', rrna_18se_seq, triphosphate=False)
    its_1_frag1_n, base_counts_its_1_frag1 = make_rrna_metabolite('its_1_frag1', its_1_frag1_seq, triphosphate=False)

    rrna_18se_formation = cobra.Reaction('FORMATION_RRNA_18se')
    rrna_18se_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site E
    rxn[rrna_21sc_n], rxn[rrna_18se_n], rxn[its_1_frag1_n] = -1, 1, 1
    rrna_18se_formation.add_metabolites(rxn)
    rrna_18se_formation.gene_reaction_rule = UTP24[0]
    its_1_frag1_degradation = fragment_degradation(its_1_frag1_n, base_counts_its_1_frag1, its_1_frag1_seq, name = 'its_1_frag1', triphosphate = False)
    
    # 18se nuclear processing------------------------------------------------------------------------------------
    rrna_18se_processed_seq = rrna_18se_seq[:-int(0.75*e_index)] # degradation of 60/80 bps of ITS1 by PARN
    rrna_18se_processed_seq += 'U'*int(0.125*e_index)+'A'*int(0.125*e_index) # polyU by PAPD5
    deg_seq = rrna_18se_seq[-int(0.75*e_index):]

    rrna_18se_processed_n, base_counts_rrna_18se_processed = make_rrna_metabolite('18se_processed', rrna_18se_processed_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_18se_processing = cobra.Reaction('PROCESSING_RRNA_18se')
    rrna_18se_processing.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[rrna_18se_n], rxn[rrna_18se_processed_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)
    # polyU/A tail synthesis
    rxn[atp_n], rxn[ntp_map_n['U']], rxn[ppi_n] = -int(0.125*e_index), -int(0.125*e_index),int(0.25*e_index)

    rrna_18se_processing.add_metabolites(rxn)
    rrna_18se_processing.gene_reaction_rule = ' and '.join(PARN + PAPD5)

    # 18se nucleocytoplasmic export-----------------------------------------------------------------------
    rrna_18se_processed_c = rrna_18se_processed_n.copy()
    rrna_18se_processed_c.id = rrna_18se_processed_c.id.replace('[n]', '[c]')
    rrna_18se_processed_c.compartment = 'c'

    rrna_18se_transport = cobra.Reaction('RRNA_18se_tn')
    rrna_18se_transport.subsytem = 'Ribosome_Biogenesis'
    rrna_18se_transport.name = 'rRNA18se nuclear export'
    rrna_18se_transport.add_metabolites({rrna_18se_processed_n: -1, rrna_18se_processed_c: 1})
    rrna_18se_transport.gene_reaction_rule = 'nucleocytoplasmic_export'
    
    # 18s formation------------------------------------------------------------------------------------
    its_1_frag2_seq = its_1_seq[:int(0.25*e_index)] + 'U'*int(0.125*e_index)+'A'*int(0.125*e_index)
    rrna_18s_c, base_counts_rrna_18s = make_rrna_metabolite('18s', rrna_18s_seq, triphosphate=False, compartment='c')
    its_1_frag2_c, base_counts_its_1_frag2 = make_rrna_metabolite('its_1_frag2', its_1_frag2_seq, triphosphate=False, compartment='c')

    rrna_18s_formation = cobra.Reaction('FORMATION_RRNA_18s')
    rrna_18s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site 3
    rxn[rrna_18se_processed_c], rxn[rrna_18s_c], rxn[its_1_frag2_c] = -1,1,1
    rrna_18s_formation.add_metabolites(rxn)
    rrna_18s_formation.gene_reaction_rule = NOB1[0]

    its_1_frag2_degradation = fragment_degradation(its_1_frag2_c, base_counts_its_1_frag2, its_1_frag2_seq, name = 'its_1_frag2', triphosphate = False, nucleus = False)
    
    # 32S formation------------------------------------------------------------------------------------
    deg_seq = its_1_seq[site_2_index:]
    rrna_32s_seq = rrna_32_5s_seq[len(deg_seq):]

    rrna_32s_n, base_counts_rrna_32s = make_rrna_metabolite('32s', rrna_32s_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_32s_formation = cobra.Reaction('FORMATION_RRNA_32s')
    rrna_32s_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_32_5s_n], rxn[rrna_32s_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)

    rrna_32s_formation.add_metabolites(rxn)
    rrna_32s_formation.gene_reaction_rule = lariat_machinery["5' Degradation"][0]
    
    #32s-->12s + 28.5s------------------------------------------------------------------------------------
 
    rrna_12s_seq = rrna_5_8s_seq + its_2_seq[:site_4_index]
    rrna_28_5s_seq = its_2_seq[site_4_index:] + rrna_28s_seq
    rrna_12s_n, base_counts_rrna_12s = make_rrna_metabolite('12s', rrna_12s_seq, triphosphate=False, compartment='n')
    rrna_28_5s_n, base_counts_rrna_28_5s = make_rrna_metabolite('28_5s', rrna_28_5s_seq, triphosphate=False, compartment='n')

    rrna_12s_28_5s_formation = cobra.Reaction('FORMATION_RRNA_12s_28_5s')
    rrna_12s_28_5s_formation.subsytem = 'Ribosome_Biogenesis'
    rxn = dict()
    rxn[h2o_n] = -1 # endonuclolytic cleavage event at site 4
    rxn[rrna_32s_n], rxn[rrna_12s_n], rxn[rrna_28_5s_n] = -1,1,1
    rrna_12s_28_5s_formation.add_metabolites(rxn)
    rrna_12s_28_5s_formation.gene_reaction_rule = LAS1[0]
    
    #28s formation------------------------------------------------------------------------------------
    deg_seq = rrna_28_5s_seq[:rrna_28_5s_seq.index(rrna_28s_seq)]

    rrna_28s_n, base_counts_rrna_28s = make_rrna_metabolite('28s', rrna_28s_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_28s_formation = cobra.Reaction('FORMATION_RRNA_28s')
    rrna_28s_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_28_5s_n], rxn[rrna_28s_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)

    rrna_28s_formation.add_metabolites(rxn)
    rrna_28s_formation.gene_reaction_rule = lariat_machinery["5' Degradation"][0]
    
    #28s export------------------------------------------------------------------------------------
    
    rrna_28s_c = rrna_28s_n.copy()
    rrna_28s_c.id = rrna_28s_c.id.replace('[n]', '[c]')
    rrna_28s_c.compartment = 'c'

    rrna_28s_transport = cobra.Reaction('RRNA_28s_tn')
    rrna_28s_transport.subsytem = 'Ribosome_Biogenesis'
    rrna_28s_transport.name = 'rRNA28s nuclear export'
    rrna_28s_transport.add_metabolites({rrna_28s_n: -1, rrna_28s_c: 1})
    rrna_28s_transport.gene_reaction_rule = 'nucleocytoplasmic_export'
    
    #7s formation------------------------------------------------------------------------------------
    deg_seq = its_2_seq[seven_s_idx: site_4_index]
    rrna_7s_seq = rrna_5_8s_seq + its_2_seq[:seven_s_idx]
    rrna_7s_n, base_counts_rrna_7s = make_rrna_metabolite('7s', rrna_7s_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_7s_formation = cobra.Reaction('FORMATION_RRNA_7s')
    rrna_7s_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_12s_n], rxn[rrna_7s_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)

    rrna_7s_formation.add_metabolites(rxn)
    rrna_7s_formation.gene_reaction_rule = ' and '.join(DIS3 + ISG20L2)
    
    #5.8s+40 formation------------------------------------------------------------------------------------
    deg_seq = its_2_seq[five_eight_plus_forty_idx: seven_s_idx]
    rrna_5_8s_plus_40_seq = rrna_5_8s_seq + its_2_seq[:five_eight_plus_forty_idx]
    rrna_5_8s_plus_40_n, base_counts_rrna_5_8s_plus_40 = make_rrna_metabolite('5_8s_plus_40', rrna_5_8s_plus_40_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_5_8s_plus_40_formation = cobra.Reaction('FORMATION_RRNA_5_8s_plus_40')
    rrna_5_8s_plus_40_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_7s_n], rxn[rrna_5_8s_plus_40_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)

    rrna_5_8s_plus_40_formation.add_metabolites(rxn)
    rrna_5_8s_plus_40_formation.gene_reaction_rule = ' and '.join(DIS3 + ISG20L2)
    
    #6s export------------------------------------------------------------------------------------

    deg_seq = its_2_seq[six_s_index: five_eight_plus_forty_idx]
    rrna_6s_seq = rrna_5_8s_seq + its_2_seq[:six_s_index]
    rrna_6s_n, base_counts_rrna_6s = make_rrna_metabolite('6s', rrna_6s_seq, triphosphate=False)
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_6s_formation = cobra.Reaction('FORMATION_RRNA_6s')
    rrna_6s_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_5_8s_plus_40_n], rxn[rrna_6s_n] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_n.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_n] = -len(deg_seq)
    rxn[h_n] = len(deg_seq)

    rrna_6s_formation.add_metabolites(rxn)
    rrna_6s_formation.gene_reaction_rule = exosome[exosome['Approved symbol'] == 'EXOSC10']['HGNC ID (gene)'].tolist()[0]

    #6s transport------------------------------------------------------------------------------------
    rrna_6s_c = rrna_6s_n.copy()
    rrna_6s_c.id = rrna_6s_c.id.replace('[n]', '[c]')
    rrna_6s_c.compartment = 'c'

    rrna_6s_transport = cobra.Reaction('RRNA_6s_tn')
    rrna_6s_transport.subsytem = 'Ribosome_Biogenesis'
    rrna_6s_transport.name = 'rRNA6s nuclear export'
    rrna_6s_transport.add_metabolites({rrna_6s_n: -1, rrna_6s_c: 1})
    rrna_6s_transport.gene_reaction_rule = 'nucleocytoplasmic_export'
    
    #5.8s formation------------------------------------------------------------------------------------

    deg_seq = its_2_seq[:six_s_index]
    rrna_5_8s_c, base_counts_rrna_5_8s = make_rrna_metabolite('5_8s', rrna_5_8s_seq, triphosphate=False, compartment = 'c')
    base_counts_deg, elements_deg = get_base_counts_and_elements(deg_seq)

    rrna_5_8s_formation = cobra.Reaction('FORMATION_RRNA_5_8s')
    rrna_5_8s_formation.subsytem = 'Ribosome_Biogenesis'

    rxn = dict()
    rxn[rrna_6s_c], rxn[rrna_5_8s_c] = -1,1
    # exonucleolytic cleavage
    for k,v in nmp_map_c.items():
        rxn[v] = base_counts_deg[k]
    rxn[h2o_c] = -len(deg_seq)
    rxn[h_c] = len(deg_seq)

    rrna_5_8s_formation.add_metabolites(rxn)
    rrna_5_8s_formation.gene_reaction_rule = ERI1[0]
    
    
    #------------------------------------------------------------------------------------
    all_reactions = [rrna_47s_transcription, rrna_45s_formation, ets_3_degradation, ets_5_frag1_degradation, 
                     rrna_30s_formation, rrna_26s_formation, ets_5_frag3_degradation, rrna_21s_formation, 
                     ets_5_frag4_degradation, rrna_21sc_formation, rrna_18se_formation, its_1_frag1_degradation, 
                     rrna_18se_processing, rrna_18se_transport, rrna_18s_formation, its_1_frag2_degradation, 
                     rrna_32s_formation, rrna_12s_28_5s_formation, rrna_28s_formation, rrna_28s_transport, 
                    rrna_7s_formation, rrna_5_8s_plus_40_formation, rrna_6s_formation, rrna_6s_transport, 
                    rrna_5_8s_formation]
    

    return all_reactions

In [15]:
def build_rrna_reactions():
    '''Reactions associated with ribosomal RNA biogenesis.'''
#     rrna5s_transcription, rrna5s_processing, rrna5s_transport, rrna5s_degradation = build_rrna5s_reactions()
    rrna5s_reactions = list(build_rrna5s_reactions())
    other_rrna_reactions = build_other_rrna_reactions()
    return rrna5s_reactions + other_rrna_reactions

def build_ribosome_protein_expression_reactions():
    '''Reactions associated with transcription and translation of ribosomal proteins'''
    
    # use this source: 
    # 1) https://www.ncbi.nlm.nih.gov/pmc/articles/PMC155282/
    # degradation - RNAP1 section last paragraph: https://www.cell.com/fulltext/S0092-8674(09)00067-1

def build_ribosome():
    '''Reactions associated with complex formation of ribosome.'''
    
    # use this source: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2174363/
    # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4361047/
    # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6315592/#B37-biomolecules-08-00123
    
    #5s and L5 complex in cytoplasm-->nuclear import

In [18]:
build_rrna_reactions()

[<Reaction TRANSCRIPTION_PRE_RRNA5s at 0x7ffbb34f1ac8>,
 <Reaction PROCESSING_RRNA5s at 0x7ffbb35033c8>,
 <Reaction RRNA5stn at 0x7ffbb3503d30>,
 <Reaction 5s_rRNA_DEGRADATION at 0x7ffbb3503ef0>,
 <Reaction TRANSCRIPTION_RRNA_47s at 0x7ffbb35155c0>,
 <Reaction FORMATION_RRNA_45s at 0x7ffbb35159b0>,
 <Reaction ets_3_DEGRADATION at 0x7ffbb3515c50>,
 <Reaction ets_5_frag1_DEGRADATION at 0x7ffbb35212b0>,
 <Reaction FORMATION_RRNA_30s_32_5s at 0x7ffbb3521da0>,
 <Reaction FORMATION_RRNA_26s at 0x7ffbb3536438>,
 <Reaction ets_5_frag3_DEGRADATION at 0x7ffbb3536320>,
 <Reaction FORMATION_RRNA_21s at 0x7ffbb3536fd0>,
 <Reaction ets_5_frag4_DEGRADATION at 0x7ffbb3536f60>,
 <Reaction FORMATION_RRNA_21sc at 0x7ffbb3549be0>,
 <Reaction FORMATION_RRNA_18se at 0x7ffbb355b550>,
 <Reaction its_1_frag1_DEGRADATION at 0x7ffbb355b470>,
 <Reaction PROCESSING_RRNA_18se at 0x7ffbb355ba20>,
 <Reaction RRNA_18se_tn at 0x7ffbb356ca58>,
 <Reaction FORMATION_RRNA_18s at 0x7ffbb356c7b8>,
 <Reaction its_1_frag2_DEGR